# Pipeline 1: Donor Churn Prediction

## 1. Problem Framing

**Business Question:** Which donors are at risk of lapsing — and what factors drive that risk?

**Who cares:** Leadership and outreach staff. From the case: *"They lose donors and don't always understand why... They want to know which donors might give more if asked, which ones are at risk of lapsing, and how to personalize outreach without a dedicated marketing team."* A churn risk score allows the small team to prioritize limited outreach time toward the donors most likely to be lost.

**Approach — Predictive (Classification):** This is a **predictive** modeling goal. We want reliable churn probability estimates per donor to drive action — not to establish causal mechanisms. Per the textbook (Ch. 1): *"Predictive modeling aims to generate reliable predictions judged by performance on new, unseen data."* Out-of-sample AUC-ROC is the primary success metric; coefficient interpretability is secondary.

**Target Variable:** `churned` — engineered binary label. A donor is churned if their `status == 'Inactive'` OR their last recorded donation was more than 365 days before the reference date.

**Success Metrics:**
- **Primary:** AUC-ROC (threshold-independent, handles class imbalance)
- **Secondary:** F1 on the churned class, Precision-Recall curve
- **Baseline:** predict-majority-class (all retained)

## 2. Data Acquisition, Preparation & Exploration

In [ ]:
import sys
sys.path.insert(0, '..')

from pyLibrary import (
    univariate, unistats, bivariate, correlation_heatmap,
    missing_data_diagnostics, missing_data_clean, basic_wrangling,
    transform_skew, cap_outliers_iqr,
    build_preprocessor, make_pipeline_for_model, split_data,
    eval_classification, plot_roc_curve, plot_confusion_matrix, plot_precision_recall,
    cross_validate_model, plot_learning_curve, plot_validation_curve,
    tune_grid, tune_random,
    select_features_filter, select_features_rfe,
    permutation_importance_report, feature_importance_plot, plot_logit_coefficients
)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../data/lighthouse_csv_v7')
REFERENCE_DATE = pd.Timestamp('2026-04-01')
CHURN_DAYS     = 365
RANDOM_STATE   = 42

In [ ]:
# ── Load raw data ─────────────────────────────────────────────────────────────
supporters = pd.read_csv(DATA_DIR / 'supporters.csv',
                         parse_dates=['created_at', 'first_donation_date'])
donations  = pd.read_csv(DATA_DIR / 'donations.csv',
                         parse_dates=['donation_date'])

print(f'Supporters: {supporters.shape}')
print(f'Donations:  {donations.shape}')
supporters.head(3)

In [ ]:
# ── Missing data diagnostics (Ch. 7) ─────────────────────────────────────────
print('=== Supporters missing data ===')
missing_data_diagnostics(supporters, verbose=True)

print('\n=== Donations missing data ===')
missing_data_diagnostics(donations, verbose=True)

In [ ]:
# ── Feature engineering: RFM + churn label ───────────────────────────────────
# Recency-Frequency-Monetary features per supporter (joined from donations)
don_stats = (
    donations.groupby('supporter_id')
    .agg(
        donation_count       = ('donation_id',      'count'),
        last_donation_date   = ('donation_date',     'max'),
        total_value          = ('estimated_value',   'sum'),
        avg_value            = ('estimated_value',   'mean'),
        is_recurring         = ('is_recurring',      lambda x: x.astype(str).str.lower().eq('true').any()),
        unique_campaigns     = ('campaign_name',     'nunique'),
        unique_channels      = ('channel_source',    'nunique'),
    )
    .reset_index()
)

df = supporters.merge(don_stats, on='supporter_id', how='left')

# Recency: days since last donation; NaN (never donated) = 9999
df['recency_days']  = (REFERENCE_DATE - df['last_donation_date']).dt.days.fillna(9999)
df['tenure_days']   = (REFERENCE_DATE - df['created_at']).dt.days

# Fill donation stats for supporters with no donations
for col in ['donation_count', 'total_value', 'avg_value', 'unique_campaigns', 'unique_channels']:
    df[col] = df[col].fillna(0)
df['is_recurring'] = df['is_recurring'].fillna(False).astype(int)

# Binary churn label
df['churned'] = ((df['status'] == 'Inactive') | (df['recency_days'] > CHURN_DAYS)).astype(int)

print(f'Total supporters: {len(df)}')
print(f'Churned: {df["churned"].sum()} ({df["churned"].mean():.1%})')
df[['supporter_id','display_name','status','recency_days','donation_count','churned']].head(6)

In [ ]:
# ── Build modeling DataFrame ──────────────────────────────────────────────────
FEATURE_COLS = [
    'recency_days', 'donation_count', 'total_value', 'avg_value',
    'tenure_days', 'unique_campaigns', 'unique_channels', 'is_recurring',
    'supporter_type', 'relationship_type', 'acquisition_channel',
    'churned'
]
model_df = df[FEATURE_COLS].copy()

# Basic wrangling: drop constant/all-unique cols (Ch. 7)
model_df = basic_wrangling(model_df, messages=True)

In [ ]:
# ── Univariate stats summary (Ch. 6) ─────────────────────────────────────────
print('=== Univariate Statistics ===')
unistats(model_df)

In [ ]:
# ── Univariate distribution plots for numeric features (Ch. 6) ───────────────
# univariate() plots box+histogram for numeric, countplot for categorical
num_cols_to_plot = ['recency_days', 'donation_count', 'total_value', 'tenure_days']
_ = univariate(model_df[num_cols_to_plot])

In [ ]:
# ── Skewness reduction (Ch. 7) ────────────────────────────────────────────────
# transform_skew tries identity, Yeo-Johnson, log1p, sqrt, cbrt; picks best
skew_cols = ['recency_days', 'donation_count', 'total_value', 'avg_value', 'tenure_days']
model_df_skew = transform_skew(model_df, features=skew_cols, suffix='_t')

# Replace originals with transformed in model_df
for col in skew_cols:
    model_df[col] = model_df_skew[col + '_t']
    
print('Skew transformations applied.')

In [ ]:
# ── IQR outlier capping (Ch. 7) ───────────────────────────────────────────────
model_df = cap_outliers_iqr(model_df, cols=skew_cols)
print('Outlier capping applied.')

In [ ]:
# ── Bivariate analysis against target (Ch. 8) ─────────────────────────────────
corr_results = bivariate(model_df.drop(columns=['supporter_type','relationship_type',
                                                  'acquisition_channel']),
                          target='churned')
print('\nPearson correlations with churn:')
print(corr_results)

In [ ]:
# ── Correlation heatmap (Ch. 8) ───────────────────────────────────────────────
num_only = model_df.select_dtypes(include='number')
corr_matrix = correlation_heatmap(num_only)

## 3. Modeling & Feature Selection

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# ── Leakage-free train/test split (Ch. 11, 15) ──────────────────────────────
X_train, X_test, y_train, y_test = split_data(
    model_df, target='churned',
    test_size=0.2, random_state=RANDOM_STATE, stratify=True
)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Churn rate — Train: {y_train.mean():.2%} | Test: {y_test.mean():.2%}')

In [ ]:
# ── Build leakage-free pipelines using build_preprocessor (Ch. 11) ───────────
# build_preprocessor infers num/cat cols and creates median+scale | mode+onehot
lr_pipe  = make_pipeline_for_model(X_train, LogisticRegression(max_iter=1000, C=0.5, random_state=RANDOM_STATE))
rf_pipe  = make_pipeline_for_model(X_train, RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE))
gb_pipe  = make_pipeline_for_model(X_train, GradientBoostingClassifier(n_estimators=200, random_state=RANDOM_STATE))

# ── Cross-validated comparison (Ch. 15) ─────────────────────────────────────
print('=== 5-Fold Stratified CV — AUC-ROC ===')
for name, pipe in [('Logistic Regression', lr_pipe),
                   ('Random Forest',       rf_pipe),
                   ('Gradient Boosting',   gb_pipe)]:
    print(f'\n{name}:')
    cross_validate_model(pipe, X_train, y_train, cv=5, scoring='roc_auc', stratified=True)

In [ ]:
# ── Learning curve: diagnose bias/variance (Ch. 15) ──────────────────────────
print('Learning Curve — Random Forest:')
plot_learning_curve(rf_pipe, X_train, y_train, cv=5, scoring='roc_auc')

In [ ]:
# ── Validation curve: tune max_depth (Ch. 15) ────────────────────────────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

preprocessor, _, _ = build_preprocessor(X_train)
rf_for_vc = Pipeline([
    ('preprocess', preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE))
])
plot_validation_curve(
    rf_for_vc, X_train, y_train,
    param_name='model__max_depth',
    param_range=[3, 5, 7, 10, 15, None],
    cv=5, scoring='roc_auc'
)

In [ ]:
# ── Hyperparameter tuning: GridSearchCV (Ch. 15) ─────────────────────────────
param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth':    [5, 10, None],
    'model__min_samples_leaf': [1, 3]
}
best_rf, gs = tune_grid(rf_pipe, param_grid, X_train, y_train,
                         cv=5, scoring='roc_auc')

In [ ]:
# ── Feature selection: RFECV (Ch. 16) ─────────────────────────────────────────
# Preprocess first, then run RFECV on the transformed array
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold

preprocessor_fit, num_cols, cat_cols = build_preprocessor(X_train)
X_train_prep = preprocessor_fit.fit_transform(X_train)
X_test_prep  = preprocessor_fit.transform(X_test)

feature_names = (
    num_cols +
    list(preprocessor_fit.named_transformers_['cat']
         .named_steps['onehot']
         .get_feature_names_out(cat_cols))
)

rfecv_est, selected = select_features_rfe(
    X_train_prep, y_train, feature_names,
    estimator=GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE),
    cv=5, scoring='roc_auc'
)

In [ ]:
# ── MDI feature importance from best RF model ─────────────────────────────────
best_rf.fit(X_train, y_train)
feature_importance_plot(
    best_rf.named_steps['model'],
    feature_names,
    top_n=15,
    title='MDI Feature Importance — Donor Churn (Random Forest)'
)

In [ ]:
# ── Permutation feature importance on TEST data (most trustworthy, Ch. 16) ───
pfi_df = permutation_importance_report(
    best_rf, X_test_prep, y_test, feature_names,
    n_repeats=10, scoring='roc_auc', top_n=15
)
print('\nTop 10 by permutation importance:')
print(pfi_df.head(10).to_string(index=False))

## 4. Evaluation & Interpretation

In [ ]:
# ── Final model evaluation on hold-out test set (Ch. 15) ─────────────────────
results = eval_classification(
    'Random Forest (Tuned)', best_rf,
    X_train, y_train, X_test, y_test
)

In [ ]:
# ── ROC Curve ────────────────────────────────────────────────────────────────
plot_roc_curve(best_rf, X_test, y_test, title='ROC Curve — Donor Churn Prediction')

In [ ]:
# ── Confusion Matrix ─────────────────────────────────────────────────────────
plot_confusion_matrix(best_rf, X_test, y_test,
                      labels=[0, 1],
                      title='Confusion Matrix — Donor Churn')

In [ ]:
# ── Precision-Recall Curve ───────────────────────────────────────────────────
plot_precision_recall(best_rf, X_test, y_test,
                      title='Precision-Recall — Donor Churn')

In [ ]:
# ── Logistic regression coefficients for interpretability (Ch. 13) ───────────
lr_pipe.fit(X_train, y_train)
coef_df = plot_logit_coefficients(
    lr_pipe, top_n=20,
    title='Logistic Regression Coefficients — Donor Churn'
)
print('\nTop positive (retention risk reduced) and negative (churn risk) features:')
print(coef_df.head(10).to_string(index=False))

In [ ]:
# ── Actionable output: at-risk donor scores ───────────────────────────────────
best_rf.fit(df[X_train.columns], df['churned'])
df['churn_risk_score'] = best_rf.predict_proba(df[X_train.columns])[:, 1]
df['churn_risk_tier']  = pd.cut(df['churn_risk_score'],
                                 bins=[0, 0.33, 0.66, 1.0],
                                 labels=['Low', 'Medium', 'High'])

print('Risk tier distribution:')
print(df['churn_risk_tier'].value_counts())

at_risk = df[(df['churned'] == 0) & (df['churn_risk_tier'] == 'High')].sort_values(
    'churn_risk_score', ascending=False
)[['supporter_id','display_name','supporter_type','acquisition_channel',
   'recency_days','donation_count','churn_risk_score']]
print(f'\nTop at-risk active donors to contact ({len(at_risk)} total):')
print(at_risk.head(10).to_string(index=False))

**Business Interpretation:**

- **Recency is the dominant predictor** — consistent with RFM donor retention theory. The longer since a donor's last gift, the higher the risk. This is partially tautological (we define churn by recency), but it is operationally valid: recency is the right trigger for outreach.
- **Frequency (donation_count) matters significantly** — first-time donors are far more likely to churn than repeat donors. Priority action: convert new one-time donors to a second gift within 90 days.
- **Recurring donors (is_recurring=1) have drastically lower churn** — this is one of the few findings where a causal interpretation is defensible: the auto-payment mechanism removes re-decision friction. Enrolling donors in recurring giving is the highest-leverage retention strategy.
- **Acquisition channel differences** — some channels produce more loyal donors. This should inform future outreach budget allocation.

**Error cost asymmetry:**
- False positive (flagging retained donor as at-risk): Low cost — unnecessary outreach message.
- False negative (missing a churning donor): High cost — permanent donor loss.
- Staff should tune the decision threshold toward higher recall (catch more churners at cost of some false alarms).

## 5. Causal and Relationship Analysis

**Relationships confirmed in data:**

| Feature | Direction | Causal Defensibility |
|---|---|---|
| Recency (days since last donation) | Higher → more churn | Partially tautological; use as trigger not cause |
| Donation frequency | Higher → less churn | Observational; first-time donors may differ systematically |
| Is recurring | Recurring → far less churn | **Most defensible causal claim**: mechanism is clear (auto-payment removes friction) |
| Acquisition channel | Varies | Cannot claim causally; confounded by content and audience differences |
| Tenure | Longer → somewhat less churn | Likely survivorship bias — long-tenure donors self-selected to stay |

**Key causal limitation:** We cannot determine *why* donors leave from this data alone — no exit survey or lapse-reason field exists. The model identifies *who* is at risk, not *what* to say to retain them. Pairing these scores with personalized outreach conversations would improve both retention and future model quality.

**Most defensible recommendation:** Establish a recurring giving program. The data shows is_recurring is among the strongest protective features and the mechanism is clear and actionable.

## 6. Deployment Notes

**Web app integration:**
- `GET /api/ml/donor-churn-risk` — returns churn risk scores for all supporters
- `GET /api/ml/donor-churn-risk/{supporterId}` — score for one supporter
- **Donors & Contributions page:** Color-coded risk badge (Low / Medium / High) next to each donor row
- **Dashboard:** "At-Risk Donors" widget showing top 5 High-risk donors needing outreach

**Retraining:** Monthly, as new donation data accumulates. Use `compute_psi()` from pyLibrary to detect distribution drift between training and production data.

**Model artifact location:** `ml-pipelines/donor_churn_model.sav`

In [ ]:
import joblib

# Refit on full dataset before saving
best_rf.fit(df[X_train.columns], df['churned'])
joblib.dump(best_rf, 'donor_churn_model.sav')

# Export risk scores for backend seeding
df[['supporter_id','display_name','status','churn_risk_score','churn_risk_tier']]\
  .to_csv('donor_churn_scores.csv', index=False)

print('Model saved: donor_churn_model.sav')
print('Scores saved: donor_churn_scores.csv')